# Baseline Gemini Muay Thai Strike Classifier

This notebook evaluates Gemini 2.5 Flash directly (without DSPy) to serve as a baseline comparison for the DSPy classifier. It uses the same data splits and evaluation metrics.


## Section 1: Setup and Configuration


In [1]:
import google.generativeai as genai
import json
from pathlib import Path
import os
from dotenv import load_dotenv
import random
import time
from typing import List, Dict, Tuple
import re

# Load environment variables
load_dotenv()

# Get Gemini API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in environment variables. Please set it in .env file.")

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)

# Initialize Gemini model
try:
    model = genai.GenerativeModel("gemini-2.5-flash")
    print('Using gemini-2.5-flash')
except:
    raise ValueError('failed to load gemini-2.5-flash')
    

# Data paths
DATA_ROOT = Path("data/combos")
SHADOW_DIR = DATA_ROOT / "shadow"
HEAVY_BAG_DIR = DATA_ROOT / "heavy_bag"

# Validation split size (same as DSPy notebook)
VAL_SPLIT_SIZE = 7

# Rate limiting for free tier (10 requests per minute)
# Use 7 seconds between requests to stay safely under limit
REQUEST_DELAY = 7

print("Setup complete!")
print(f"Data directories: {SHADOW_DIR}, {HEAVY_BAG_DIR}")
print(f"Rate limiting: {REQUEST_DELAY}s delay between API calls")


Using gemini-2.5-flash
Setup complete!
Data directories: data\combos\shadow, data\combos\heavy_bag
Rate limiting: 7s delay between API calls


## Section 2: Data Loading


In [2]:
def load_labels(json_path: Path) -> Dict[str, List[str]]:
    """Load true labels from JSON file."""
    with open(json_path, 'r') as f:
        return json.load(f)

def create_examples(labels: Dict[str, List[str]], video_dir: Path, dataset_name: str) -> List[Dict]:
    """Create example dictionaries from labels and video directory."""
    examples = []
    for video_id, strikes in labels.items():
        video_path = video_dir / f"{video_id}.mov"
        if video_path.exists():
            example = {
                'video_path': str(video_path),
                'strikes': strikes
            }
            examples.append(example)
        else:
            print(f"Warning: Video {video_path} not found")
    return examples

# Load labels from both datasets
shadow_labels = load_labels(SHADOW_DIR / "true_labels.json")
heavy_bag_labels = load_labels(HEAVY_BAG_DIR / "true_labels.json")

# Create examples
shadow_examples = create_examples(shadow_labels, SHADOW_DIR, "shadow")
heavy_bag_examples = create_examples(heavy_bag_labels, HEAVY_BAG_DIR, "heavy_bag")

print(f"Loaded {len(shadow_examples)} shadow examples")
print(f"Loaded {len(heavy_bag_examples)} heavy_bag examples")


Loaded 20 shadow examples
Loaded 19 heavy_bag examples


In [3]:
# Split data into train and validation
# Use the same random seed (42) as DSPy notebook for reproducibility
# Flipped split: smaller training set, larger validation set (matching DSPy recommendation)

random.seed(42)  # Same seed as DSPy notebook

# Shadow split
shadow_indices = list(range(len(shadow_examples)))
random.shuffle(shadow_indices)
shadow_train_indices = set(shadow_indices[:VAL_SPLIT_SIZE])
shadow_train = [shadow_examples[i] for i in shadow_train_indices]
shadow_val = [shadow_examples[i] for i in range(len(shadow_examples)) if i not in shadow_train_indices]

# Heavy bag split
heavy_bag_indices = list(range(len(heavy_bag_examples)))
random.shuffle(heavy_bag_indices)
heavy_bag_train_indices = set(heavy_bag_indices[:VAL_SPLIT_SIZE])
heavy_bag_train = [heavy_bag_examples[i] for i in heavy_bag_train_indices]
heavy_bag_val = [heavy_bag_examples[i] for i in range(len(heavy_bag_examples)) if i not in heavy_bag_train_indices]

# Combine train and validation sets
train_examples = shadow_train + heavy_bag_train
val_examples = shadow_val + heavy_bag_val

print(f"Training set: {len(train_examples)} examples ({len(shadow_train)} shadow + {len(heavy_bag_train)} heavy_bag)")
print(f"Validation set: {len(val_examples)} examples ({len(shadow_val)} shadow + {len(heavy_bag_val)} heavy_bag)")

# Display some statistics
all_strikes = set()
for ex in train_examples + val_examples:
    all_strikes.update(ex['strikes'])

print(f"\nUnique strike types: {sorted(all_strikes)}")
print(f"Total unique strike types: {len(all_strikes)}")


Training set: 14 examples (7 shadow + 7 heavy_bag)
Validation set: 25 examples (13 shadow + 12 heavy_bag)

Unique strike types: ['cross', 'jab', 'knee', 'lead_elbow', 'lead_hook', 'lead_uppercut', 'lowkick', 'rear_elbow', 'rear_hook', 'roundhouse', 'switch_kick', 'switch_knee', 'teep']
Total unique strike types: 13


## Section 3: Direct Gemini Classification


In [ ]:
def classify_video_baseline(video_path: str, model=model) -> List[str]:
    """
    Classify Muay Thai strikes in a video using Gemini API directly.
    Returns a list of strike names in sequence.
    """
    prompt = """Watch this Muay Thai training video and identify all strikes in the order they appear. 

    Available strike types with detailed descriptions:
- jab: A quick, straight forward punch with the lead (front) hand, arm extends straight forward from shoulder
- cross: A powerful straight punch with the rear hand, arm extends straight forward from shoulder, body rotates
- lead_hook: A curved punch with the lead hand, arm swings in an arc from the side
- rear_hook: A curved punch with the rear hand, arm swings in an arc from the side
- lead_uppercut: An upward punch with the lead hand, arm moves upward from below
- rear_uppercut: An upward punch with the rear hand, arm moves upward from below
- roundhouse: A circular kick where the leg swings in an arc, typically hitting with the shin or instep. The leg comes from the side and wraps around. Also called "low kick" when targeting the leg.
- switch_kick: A roundhouse kick performed after switching stance (stepping forward with rear leg first)
- teep: A front push kick where the foot pushes straight forward, like a front kick. The leg extends forward, not sideways. This is DIFFERENT from a roundhouse kick.
- knee: A knee strike where the knee is driven upward into a target
- switch_knee: A knee strike after switching stance
- lead_elbow: An elbow strike with the lead arm, elbow strikes forward or sideways
- rear_elbow: An elbow strike with the rear arm, elbow strikes forward or sideways

Return your answer as a JSON array of strike names, e.g. ["jab", "cross", "lead_hook"].
Only return the JSON array, nothing else."""
    
    try:
        # Load video file
        video_file = genai.upload_file(path=video_path)
        
        # Retrieve the file to check its state
        myfile = genai.get_file(name=video_file.name)
        
        # Wait for the file to be in an ACTIVE state
        while myfile.state.name != 'ACTIVE':
            print(f'File state is {myfile.state.name}. Waiting...')
            time.sleep(5)  # Wait for 5 seconds before checking again
            myfile = genai.get_file(name=video_file.name)  # Get the updated file status
        
        print(f'File {myfile.name} is now {myfile.state.name}.')
        
        # Generate content
        response = model.generate_content([prompt, video_file])
        
        # Parse response
        response_text = response.text.strip()
        
        # Try to extract JSON array from response
        # First, try direct JSON parsing
        try:
            strikes = json.loads(response_text)
            if isinstance(strikes, list):
                return strikes
        except json.JSONDecodeError:
            pass
        
        # If direct parsing fails, try to find JSON array in the text
        json_match = re.search(r'\[.*?\]', response_text)
        if json_match:
            try:
                strikes = json.loads(json_match.group())
                if isinstance(strikes, list):
                    return strikes
            except json.JSONDecodeError:
                pass
        
        # If still no luck, return empty list
        print(f"Warning: Could not parse JSON from response: {response_text[:100]}")
        return []
        
    except Exception as e:
        print(f"Error classifying video {video_path}: {e}")
        return []

print("Baseline classification function created!")


Baseline classification function created!


## Section 4: Evaluation


In [5]:
# Evaluation metrics (same as DSPy notebook)

def exact_match_metric(true_strikes: List[str], predicted_strikes: List[str]) -> bool:
    """Evaluate exact sequence match between predicted and true strikes."""
    return predicted_strikes == true_strikes

def sequence_accuracy_metric(true_strikes: List[str], predicted_strikes: List[str]) -> float:
    """Calculate sequence accuracy (partial credit for correct strikes in order)."""
    if not predicted_strikes or not true_strikes:
        return 0.0
    
    # Calculate positional accuracy
    min_len = min(len(predicted_strikes), len(true_strikes))
    if min_len == 0:
        return 0.0
    
    matches = sum(1 for i in range(min_len) if predicted_strikes[i] == true_strikes[i])
    return matches / len(true_strikes)

print("Evaluation metrics defined!")


Evaluation metrics defined!


In [6]:
# Evaluate on validation set
print(f"Evaluating on {len(val_examples)} validation examples...")
print(f"Using {REQUEST_DELAY}s delay between requests to avoid rate limits\n")

results = []
exact_matches = 0
total_accuracy = 0.0

for i, example in enumerate(val_examples):
    try:
        # Add rate limiting delay (except for first request)
        if i > 0:
            print(f"Waiting {REQUEST_DELAY}s to avoid rate limits...")
            time.sleep(REQUEST_DELAY)
        
        print(f"Processing {i+1}/{len(val_examples)}: {Path(example['video_path']).name}")
        
        # Classify video
        predicted_strikes = classify_video_baseline(example['video_path'])
        
        true_strikes = example['strikes']
        exact_match = exact_match_metric(true_strikes, predicted_strikes)
        accuracy = sequence_accuracy_metric(true_strikes, predicted_strikes)
        
        if exact_match:
            exact_matches += 1
        total_accuracy += accuracy
        
        results.append({
            'video_path': example['video_path'],
            'true_strikes': true_strikes,
            'predicted_strikes': predicted_strikes,
            'exact_match': exact_match,
            'sequence_accuracy': accuracy
        })
        
        print(f"  True: {true_strikes}")
        print(f"  Pred: {predicted_strikes}")
        print(f"  Match: {'✓' if exact_match else '✗'} (Accuracy: {accuracy:.2%})\n")
            
    except Exception as e:
        print(f"Error processing {example['video_path']}: {e}\n")
        results.append({
            'video_path': example['video_path'],
            'true_strikes': example['strikes'],
            'predicted_strikes': [],
            'exact_match': False,
            'sequence_accuracy': 0.0,
            'error': str(e)
        })

print("Evaluation complete!")


Evaluating on 25 validation examples...
Using 7s delay between requests to avoid rate limits

Processing 1/25: 1.mov
File state is PROCESSING. Waiting...
File files/21jugu4cqw5d is now ACTIVE.
  True: ['jab', 'cross', 'lead_hook', 'roundhouse']
  Pred: ['jab', 'cross', 'right_kick']
  Match: ✗ (Accuracy: 50.00%)

Waiting 7s to avoid rate limits...
Processing 2/25: 2.mov
File state is PROCESSING. Waiting...
File files/he5mmj07fx1y is now ACTIVE.
  True: ['jab', 'cross', 'lowkick']
  Pred: ['jab']
  Match: ✗ (Accuracy: 33.33%)

Waiting 7s to avoid rate limits...
Processing 3/25: 3.mov
File state is PROCESSING. Waiting...
File files/bg95aqhyh7xt is now ACTIVE.
  True: ['lead_uppercut', 'cross', 'rear_elbow', 'knee']
  Pred: ['cross', 'lead_knee']
  Match: ✗ (Accuracy: 0.00%)

Waiting 7s to avoid rate limits...
Processing 4/25: 4.mov
File state is PROCESSING. Waiting...
File files/gy0nyzqzoahl is now ACTIVE.
  True: ['jab', 'cross', 'lead_hook', 'lead_hook', 'roundhouse']
  Pred: ['lead_te

In [7]:
# Calculate and display metrics
exact_match_accuracy = exact_matches / len(val_examples) if val_examples else 0
average_sequence_accuracy = total_accuracy / len(val_examples) if val_examples else 0

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Exact Match Accuracy: {exact_match_accuracy:.2%} ({exact_matches}/{len(val_examples)})")
print(f"Average Sequence Accuracy: {average_sequence_accuracy:.2%}")
print()

# Display some example predictions
print("Sample Predictions:")
print("-" * 60)
for i, result in enumerate(results[:5], 1):
    print(f"\nExample {i}:")
    print(f"  Video: {Path(result['video_path']).name}")
    print(f"  True:  {result['true_strikes']}")
    print(f"  Pred:  {result['predicted_strikes']}")
    print(f"  Match: {'✓' if result['exact_match'] else '✗'} (Accuracy: {result['sequence_accuracy']:.2%})")


EVALUATION RESULTS
Exact Match Accuracy: 0.00% (0/25)
Average Sequence Accuracy: 20.33%

Sample Predictions:
------------------------------------------------------------

Example 1:
  Video: 1.mov
  True:  ['jab', 'cross', 'lead_hook', 'roundhouse']
  Pred:  ['jab', 'cross', 'right_kick']
  Match: ✗ (Accuracy: 50.00%)

Example 2:
  Video: 2.mov
  True:  ['jab', 'cross', 'lowkick']
  Pred:  ['jab']
  Match: ✗ (Accuracy: 33.33%)

Example 3:
  Video: 3.mov
  True:  ['lead_uppercut', 'cross', 'rear_elbow', 'knee']
  Pred:  ['cross', 'lead_knee']
  Match: ✗ (Accuracy: 0.00%)

Example 4:
  Video: 4.mov
  True:  ['jab', 'cross', 'lead_hook', 'lead_hook', 'roundhouse']
  Pred:  ['lead_teep']
  Match: ✗ (Accuracy: 0.00%)

Example 5:
  Video: 7.mov
  True:  ['lead_uppercut', 'cross', 'lead_hook']
  Pred:  ['lead_roundhouse_kick']
  Match: ✗ (Accuracy: 0.00%)


In [8]:
# Calculate per-strike accuracy
from collections import defaultdict

strike_stats = defaultdict(lambda: {'correct': 0, 'total': 0, 'predicted': 0})

for result in results:
    true_strikes = result['true_strikes']
    pred_strikes = result['predicted_strikes']
    
    # Count true strikes
    for strike in true_strikes:
        strike_stats[strike]['total'] += 1
    
    # Count predicted strikes
    for strike in pred_strikes:
        strike_stats[strike]['predicted'] += 1
    
    # Count correct predictions at each position
    min_len = min(len(true_strikes), len(pred_strikes))
    for i in range(min_len):
        if true_strikes[i] == pred_strikes[i]:
            strike_stats[true_strikes[i]]['correct'] += 1

print("\nPer-Strike Accuracy:")
print("-" * 60)
for strike in sorted(strike_stats.keys()):
    stats = strike_stats[strike]
    accuracy = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"  {strike:20s}: {accuracy:6.2%} ({stats['correct']}/{stats['total']} correct, {stats['predicted']} predicted)")



Per-Strike Accuracy:
------------------------------------------------------------
  cross               : 25.00% (5/20 correct, 16 predicted)
  downward_elbow      :  0.00% (0/0 correct, 1 predicted)
  jab                 : 52.17% (12/23 correct, 13 predicted)
  knee                :  0.00% (0/4 correct, 0 predicted)
  lead_elbow          :  0.00% (0/1 correct, 0 predicted)
  lead_elbow_strike   :  0.00% (0/0 correct, 1 predicted)
  lead_hook           :  6.25% (1/16 correct, 5 predicted)
  lead_knee           :  0.00% (0/0 correct, 5 predicted)
  lead_knee_strike    :  0.00% (0/0 correct, 1 predicted)
  lead_roundhouse_kick:  0.00% (0/0 correct, 6 predicted)
  lead_teep           :  0.00% (0/0 correct, 1 predicted)
  lead_uppercut       :  0.00% (0/4 correct, 0 predicted)
  lowkick             :  0.00% (0/1 correct, 0 predicted)
  rear_elbow          :  0.00% (0/2 correct, 0 predicted)
  rear_hook           :  0.00% (0/2 correct, 0 predicted)
  rear_roundhouse_kick:  0.00% (0/0 corre

## Section 5: Results and Comparison


In [9]:
# Save evaluation results (same format as DSPy notebook)
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# Save evaluation results
results_data = {
    'metrics': {
        'exact_match_accuracy': exact_match_accuracy,
        'average_sequence_accuracy': average_sequence_accuracy,
        'total_examples': len(val_examples),
        'exact_matches': exact_matches
    },
    'results': results,
    'strike_stats': {k: dict(v) for k, v in strike_stats.items()}
}

with open(output_dir / "combo_baseline_evaluation_results.json", 'w') as f:
    json.dump(results_data, f, indent=2)

print(f"Results saved to {output_dir / 'combo_baseline_evaluation_results.json'}")


Results saved to results\combo_baseline_evaluation_results.json


In [10]:
# Display summary
print("\n" + "=" * 60)
print("BASELINE EVALUATION SUMMARY")
print("=" * 60)
print(f"Model: Gemini 2.5 Flash (Direct API)")
print(f"Approach: Baseline (no DSPy, no few-shot examples)")
print(f"Training examples: {len(train_examples)}")
print(f"Validation examples: {len(val_examples)}")
print(f"\nFinal Performance:")
print(f"  Exact Match Accuracy: {exact_match_accuracy:.2%}")
print(f"  Average Sequence Accuracy: {average_sequence_accuracy:.2%}")
print(f"\nResults saved to: {output_dir}/combo_baseline_evaluation_results.json")
print("\nNote: Compare these results with DSPy classifier results for evaluation.")
print("=" * 60)



BASELINE EVALUATION SUMMARY
Model: Gemini 2.5 Flash (Direct API)
Approach: Baseline (no DSPy, no few-shot examples)
Training examples: 14
Validation examples: 25

Final Performance:
  Exact Match Accuracy: 0.00%
  Average Sequence Accuracy: 20.33%

Results saved to: results/combo_baseline_evaluation_results.json

Note: Compare these results with DSPy classifier results for evaluation.
